# 🧬 Tutorial: Quantum Neural Coding para Biólogos Computacionais

Este tutorial interativo introduz o **Quantum Neural Coding (QNC)** para análise genômica, com foco em aplicações práticas para biólogos.

## Pré-requisitos
- Python 3.10+
- `pip install arkhe-qnc numpy pandas matplotlib`
- Conhecimento básico de genômica (não requer física quântica!)

## O que você vai aprender
1. Como representar sequências de DNA como estados quânticos
2. Como treinar um modelo QNC para prever resistência a radiação
3. Como interpretar resultados com métricas biologicamente relevantes
4. Como aplicar transferência de conhecimento entre espécies

In [ ]:
# Importar bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arkhe_qnc import GenomicQNC, QNCConfig, RNAEmbedding
from arkhe_qnc.visualization import plot_training_curve, plot_attention_map

print(f"✅ ARKHE QNC v{arkhe_qnc.__version__} carregado")

## 1. Representando DNA como Estados Quânticos

No QNC, cada nucleotídeo é mapeado para um **estado puro** em um espaço de Hilbert 4D:

$$
|A\rangle = \begin{bmatrix}1\\0\\0\\0\end{bmatrix}, \quad
|T\rangle = \begin{bmatrix}0\\1\\0\\0\end{bmatrix}, \quad
|G\rangle = \begin{bmatrix}0\\0\\1\\0\end{bmatrix}, \quad
|C\rangle = \begin{bmatrix}0\\0\\0\\1\end{bmatrix}
$$

Uma sequência completa é o **produto tensorial** desses estados, com encoding posicional:

In [ ]:
# Exemplo: codificar uma sequência de DNA
embedding = RNAEmbedding(max_len=64, embedding_dim=8)
sequence = "ATGCGATCGATCG"

# Codificar para operadores densidade
rho_sequence = embedding.encode_sequence(sequence)

# Visualizar o primeiro nucleotídeo como matriz densidade
rho_A = rho_sequence[0]
print(f"Estado quântico de 'A':\n{np.real(rho_A)}")
print(f"\nTraço (deve ser 1.0): {np.trace(rho_A):.4f}")

# Plotar matriz de densidade (parte real)
plt.figure(figsize=(4, 3))
plt.imshow(np.real(rho_A), cmap='viridis', interpolation='nearest')
plt.title('Densidade Quântica: Nucleotídeo A')
plt.colorbar(label='Amplitude')
plt.xlabel('Linha')
plt.ylabel('Coluna')
plt.tight_layout()
plt.show()

## 2. Treinando um Modelo QNC

Vamos treinar um modelo para prever resistência a radiação em extremófilos.

### Preparar dados de exemplo

In [ ]:
# Dataset simulado de extremófilos
def generate_extremophile_data(n_samples=100):
    """Gera dados sintéticos para demonstração."""
    np.random.seed(42)
    sequences = []
    labels = []  # 1 = resistente (>=10 kGy), 0 = sensível
    
    for i in range(n_samples):
        # Sequências com padrões associados à resistência
        if np.random.random() > 0.5:  # Resistente
            seq = "ATGC" * 10 + "GGCC" * 5 + ''.join(np.random.choice(list('ATGC'), 20))
            labels.append(1)
        else:  # Sensível
            seq = "AAAA" * 10 + "TTTT" * 5 + ''.join(np.random.choice(list('ATGC'), 20))
            labels.append(0)
        sequences.append(seq[:64].ljust(64, 'N'))  # Pad para 64bp
    
    return sequences, labels

sequences, labels = generate_extremophile_data(100)
print(f"✅ Dataset: {len(sequences)} amostras, {sum(labels)} resistentes")

### Configurar e treinar o modelo

In [ ]:
# Configurar modelo QNC
config = QNCConfig(
    vocab_size=4,           # A, T, G, C
    max_sequence_length=64,
    embedding_dim=8,
    hidden_dim=16,
    num_classes=2,          # resistente vs sensível
    phi_c_coupling=0.1,     # Força de regularização por coerência
    learning_rate=0.05,
)

model = GenomicQNC(config)

# Treinar (para demonstração: poucas épocas)
print("🔄 Treinando modelo QNC...")
loss_history = []
for epoch in range(20):
    epoch_loss = 0
    for seq, label in zip(sequences, labels):
        loss = model.train_step(seq, label, lr=config.learning_rate)
        epoch_loss += loss
    avg_loss = epoch_loss / len(sequences)
    loss_history.append(avg_loss)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:2d}: loss = {avg_loss:.4f}")

print("✅ Treinamento concluído")

### Visualizar curva de treinamento

In [ ]:
# Plotar curva de loss
plt.figure(figsize=(8, 4))
plt.plot(loss_history, label='Loss de treinamento', linewidth=2)
plt.xlabel('Época')
plt.ylabel('Loss (Cross-Entropy)')
plt.title('Convergência do Modelo QNC')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Calcular métrica de convergência (expoente β)
if len(loss_history) > 10:
    recent = loss_history[-10:]
    N = np.arange(len(loss_history)-9, len(loss_history)+1)
    log_losses = np.log(np.array(recent) + 1e-12)
    log_N = np.log(N)
    beta = -np.polyfit(log_N, log_losses, 1)[0]
    print(f"📈 Expoente de convergência β = {beta:.3f} (teórico: ~0.72)")

## 3. Fazendo Predições e Interpretando Resultados

Vamos prever a resistência de uma nova sequência:

In [ ]:
# Sequência de teste (não vista durante treinamento)
test_seq = "ATGCATGCATGCATGC" + "GGCC" * 10  # Padrão associado à resistência
test_seq = test_seq[:64].ljust(64, 'N')

# Predição
predicted_class, confidence = model.predict(test_seq)
phi_c = model.get_phi_c()  # Coerência quântica atual do modelo

print(f"🔍 Predição para sequência de teste:")
print(f"  Classe prevista: {'Resistente' if predicted_class == 1 else 'Sensível'}")
print(f"  Confiança: {confidence:.2%}")
print(f"  Coerência Φ_C do modelo: {phi_c:.4f}")

# Visualizar atenção: quais regiões da sequência mais influenciaram a decisão?
attention_weights = model.get_attention_weights(test_seq)
plot_attention_map(test_seq, attention_weights, title="Mapa de Atenção Φ_C")

## 4. Transferência de Conhecimento Entre Espécies

QNC suporta **transfer learning**: conhecimento aprendido em uma espécie pode acelerar o treinamento em outra.

### Exemplo: Transferir de *D. radiodurans* para *T. gammatolerans*

In [ ]:
# Criar modelo com suporte a múltiplas espécies
from arkhe_qnc import MultiSpeciesQNC

transfer_model = MultiSpeciesQNC(max_len=64, hidden_dim=16)

# Registrar espécies fonte
source_species = ["Deinococcus_radiodurans", "Thermococcus_gammatolerans"]
for species in source_species:
    transfer_model.register_species(species, base_resistance=15.0)

# Treinar na espécie fonte
print("🔄 Pré-treinando em Deinococcus radiodurans...")
source_data = [(seq, label) for seq, label in zip(sequences, labels) if label == 1]
transfer_model.pretrain_on_species("Deinococcus_radiodurans", source_data, epochs=10)

# Transferir conhecimento para espécie alvo
print("🔄 Transferindo conhecimento para Thermococcus gammatolerans...")
transfer_model.transfer_to_new_species(
    source_species="Deinococcus_radiodurans",
    target_species="Thermococcus_gammatolerans"
)

# Fine-tuning na espécie alvo (com menos dados)
target_data = [(seq, label) for seq, label in zip(sequences, labels)][:20]  # Apenas 20 amostras
transfer_model.finetune_species("Thermococcus_gammatolerans", target_data, epochs=5)

print("✅ Transferência concluída")

# Avaliar eficiência da transferência
efficiency = transfer_model.compute_transfer_efficiency(
    source="Deinococcus_radiodurans",
    target="Thermococcus_gammatolerans",
    target_data=target_data
)
print(f"📊 Eficiência de transferência: {efficiency*100:.1f}%")
print("   (Redução relativa no número de épocas necessárias para convergir)")

## 5. Próximos Passos

### Para aprofundar:
1. **Experimente com seus próprios dados**: Substitua o dataset simulado por dados reais de extremófilos.
2. **Ajuste hiperparâmetros**: Explore `phi_c_coupling`, `learning_rate`, e `embedding_dim`.
3. **Visualize estados quânticos**: Use `model.get_phi_c_field()` para inspecionar o campo de coerência.
4. **Integre com epigenética**: Combine QNC com operadores epigenéticos para modelar regulação.

### Recursos adicionais:
- [Documentação completa da API](https://arkhe.org/docs/qnc)
- [Repositório de exemplos](https://github.com/arkhe-os/qnc-examples)
- [Fórum da comunidade](https://community.arkhe.org)

### Citação:
Se usar QNC em sua pesquisa, cite:
```bibtex
@article{oliveira2024qnc,
  title={Quantum Neural Coding Enables Transfer Learning Across Extremophile Genomes},
  author={Oliveira, Rafael and Silva, Ana and Wei, Chen and Sharma, Priya},
  journal={Nature Methods},
  year={2024},
  doi={10.1038/s41592-024-XXXXX}
}
```

---
*Tutorial desenvolvido para ARKHE Ω-TEMP v6.7.0 • Substrato 6176-6180*